<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/selective-decoder-routing/blob/experiment%2Fdecoder-runtime-motivation/notebooks/decoder_runtime_motivation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PM versus correlated matching: measured decoder cost

Run this notebook in a **Colab CPU runtime**. It benchmarks standard PyMatching (PM) and PyMatching's two-pass correlated matching (CM) on identical experimental detector events from [Google Quantum AI's Zenodo archive](https://zenodo.org/records/6804040). No test split, trained router, or claimed router saving is involved.

The results measure **these PyMatching implementations on this Colab CPU**. Stored Google predictions are checked where available; matching them is necessary before describing the accuracy as a reproduction of the archived decoders. The archive stays in `/content`, outside Git.

## 1. Install and record the environment

This notebook pins the Stim and PyMatching versions used in the saved pilot run. Run both decoders in this same runtime; Colab CPU types can change between sessions.

In [10]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'stim==1.16.0', 'pymatching==2.4.0', 'requests'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'stim==1.16.0', 'pymatching==2.4.0', 'requests'], returncode=0)

In [11]:
import hashlib
import itertools
import os
import platform
import shutil
import time
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import numpy as np
import pandas as pd
import pymatching
import stim

WORK = Path('/content/decoder_runtime')
WORK.mkdir(parents=True, exist_ok=True)
print({'python': sys.version.split()[0], 'stim': stim.__version__,
       'pymatching': pymatching.__version__, 'numpy': np.__version__,
       'platform': platform.platform(), 'processor': platform.processor(),
       'logical_cpu_count': os.cpu_count()})
CPU_MODEL = next((line.split(':', 1)[1].strip() for line in Path('/proc/cpuinfo').read_text().splitlines() if line.startswith('model name')), 'unknown')
print('CPU model:', CPU_MODEL)

{'python': '3.13.18', 'stim': '1.16.0', 'pymatching': '2.4.0', 'numpy': '2.4.6', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'processor': 'x86_64', 'logical_cpu_count': 2}
CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz


## 2. Get and inspect the archive

The published ZIP is about 315 MB. The cell downloads it once and checks the MD5 shown on Zenodo. If downloading in Colab fails, upload the ZIP or mount Drive and set `ARCHIVE` to its location before running this cell. Internal filenames are discovered from the ZIP rather than assumed.

In [12]:
import requests
ARCHIVE = WORK / 'google_qec3v5_experiment_data.zip'
URL = 'https://zenodo.org/records/6804040/files/google_qec3v5_experiment_data.zip?download=1'
EXPECTED_MD5 = 'a7fd8b481c3087090093106382dc217d'
if not ARCHIVE.exists():
    temporary = ARCHIVE.with_suffix('.partial')
    with requests.get(URL, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        with temporary.open('wb') as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
    temporary.replace(ARCHIVE)
digest = hashlib.md5()
with ARCHIVE.open('rb') as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b''):
        digest.update(chunk)
assert digest.hexdigest() == EXPECTED_MD5, f'Archive checksum mismatch: {digest.hexdigest()}'
print('Verified archive:', ARCHIVE, 'bytes:', ARCHIVE.stat().st_size)

Verified archive: /content/decoder_runtime/google_qec3v5_experiment_data.zip bytes: 315490804


In [13]:
with ZipFile(ARCHIVE) as archive:
    members = [name for name in archive.namelist() if not name.endswith('/')]
    readmes = [name for name in members if PurePosixPath(name).name.lower() == 'readme.txt']
    if readmes:
        print('Archive README:', readmes[0])
        print(archive.read(readmes[0]).decode('utf-8', errors='replace')[:8000])
by_dir = {}
for name in members:
    path = PurePosixPath(name)
    by_dir.setdefault(str(path.parent), {})[path.name] = name
required = {'circuit_detector_error_model.dem', 'detection_events.b8'}
candidates = sorted(directory for directory, files in by_dir.items() if required <= files.keys())
print('Configurations with a DEM and detector events:', len(candidates))
for index, directory in enumerate(candidates[:40]):
    print(index, directory)
assert candidates, 'No matching DEM/detection-event pair found. Inspect ZIP member names and its README.'

Archive README: README.txt
This directory contains subdirectories each containing data files describing
quantum error correction experiments.

Each subdirectory's name follows the pattern
"{code}_b{basis}_d{distance}_r{rounds}_center_{row}_{col}".
The row and col are the coordinates of the center data qubit. The basis is
whether the X or Z logical observable was tested. The distance is the code
distance of the experiment. The number of rounds is how many times each
measurement qubit was measured (the data measurements at the end of the circuit
aren't counted as an additional round).

As an example of how these files can be used, the following bash command line
can be used to find the high energy event (noted in the paper) that increased
the logical error rate of the repetition code at distance 25:

        # Assumes a linux-like command line environment.
        # Assumes your working directory is the directory containing this readme.
        # Note the huge cluster of mismatches near 

## 3. Select one surface-code configuration

Set `CONFIG_FILTER` to part of the desired experiment path (for example a distance, basis, or round count), then choose `CONFIG_INDEX` from the printed matches. Start with one configuration; do not pool different configurations into one timing result. Only its necessary files are extracted.

In [14]:
CONFIG_FILTER = 'surface_code'
CONFIG_INDEX = 0
matches = [directory for directory in candidates if CONFIG_FILTER.lower() in directory.lower()]
print('Matching configurations:')
for index, directory in enumerate(matches[:40]):
    print(index, directory)
assert matches, 'No configuration matched. Inspect candidates and change CONFIG_FILTER.'
chosen_dir = matches[CONFIG_INDEX]
selected = by_dir[chosen_dir]
print('Selected:', chosen_dir)
print('Available files:', sorted(selected))
wanted = ['circuit_detector_error_model.dem', 'detection_events.b8',
          'obs_flips_actual.01', 'obs_flips_predicted_by_pymatching.01',
          'obs_flips_predicted_by_correlated_matching.01']
DEST = WORK / hashlib.sha256(chosen_dir.encode()).hexdigest()[:12]
DEST.mkdir(exist_ok=True)
paths = {}
with ZipFile(ARCHIVE) as archive:
    for basename in wanted:
        if basename in selected:
            destination = DEST / basename
            with archive.open(selected[basename]) as source, destination.open('wb') as output:
                shutil.copyfileobj(source, output)
            paths[basename] = destination
print('Extracted:', sorted(paths))

Matching configurations:
0 surface_code_bX_d3_r01_center_3_5
1 surface_code_bX_d3_r01_center_5_3
2 surface_code_bX_d3_r01_center_5_7
3 surface_code_bX_d3_r01_center_7_5
4 surface_code_bX_d3_r03_center_3_5
5 surface_code_bX_d3_r03_center_5_3
6 surface_code_bX_d3_r03_center_5_7
7 surface_code_bX_d3_r03_center_7_5
8 surface_code_bX_d3_r05_center_3_5
9 surface_code_bX_d3_r05_center_5_3
10 surface_code_bX_d3_r05_center_5_7
11 surface_code_bX_d3_r05_center_7_5
12 surface_code_bX_d3_r07_center_3_5
13 surface_code_bX_d3_r07_center_5_3
14 surface_code_bX_d3_r07_center_5_7
15 surface_code_bX_d3_r07_center_7_5
16 surface_code_bX_d3_r09_center_3_5
17 surface_code_bX_d3_r09_center_5_3
18 surface_code_bX_d3_r09_center_5_7
19 surface_code_bX_d3_r09_center_7_5
20 surface_code_bX_d3_r11_center_3_5
21 surface_code_bX_d3_r11_center_5_3
22 surface_code_bX_d3_r11_center_5_7
23 surface_code_bX_d3_r11_center_7_5
24 surface_code_bX_d3_r13_center_3_5
25 surface_code_bX_d3_r13_center_5_3
26 surface_code_bX_d3_r

## 4. Parse and validate the same shots for both decoders

Stim's `b8` format packs bits in little-endian bit order within each byte. Its reader uses the DEM detector count; unpack only the selected first `N_SHOTS` rows. Prediction and outcome files, if present, are read in that same row order.

In [15]:
N_SHOTS = 10_000  # Set to 1_000 for a pilot.
dem = stim.DetectorErrorModel.from_file(paths['circuit_detector_error_model.dem'])
assert dem.num_detectors > 0 and dem.num_observables > 0
packed = stim.read_shot_data_file(path=paths['detection_events.b8'], format='b8',
                                  num_detectors=dem.num_detectors, bit_packed=True)
assert packed.ndim == 2 and packed.shape[1] == (dem.num_detectors + 7) // 8
assert len(packed) >= N_SHOTS, f'Only {len(packed)} shots available; reduce N_SHOTS.'
shots = np.unpackbits(packed[:N_SHOTS], axis=1, bitorder='little')[:, :dem.num_detectors]
shots = np.ascontiguousarray(shots, dtype=np.uint8)
assert shots.shape == (N_SHOTS, dem.num_detectors)
print({'shots': len(shots), 'detectors': dem.num_detectors,
       'observables': dem.num_observables, 'DEM_correlation_separators': str(dem).count('^')})
if '^' not in str(dem):
    print('Check DEM content: no ^ separators were found, so correlation information may be absent.')

def read_01(basename):
    path = paths.get(basename)
    if path is None:
        return None
    with path.open('r', encoding='ascii') as source:
        lines = [line.strip() for line in itertools.islice(source, N_SHOTS)]
    assert len(lines) == N_SHOTS, f'{basename} has fewer than {N_SHOTS} rows'
    assert all(len(line) == dem.num_observables and set(line) <= {'0', '1'} for line in lines), basename
    return np.asarray([[int(bit) for bit in line] for line in lines], dtype=np.uint8)

actual = read_01('obs_flips_actual.01')
stored_pm = read_01('obs_flips_predicted_by_pymatching.01')
stored_cm = read_01('obs_flips_predicted_by_correlated_matching.01')
print('Reference files available:', {'actual': actual is not None, 'PM': stored_pm is not None, 'CM': stored_cm is not None})

{'shots': 10000, 'detectors': 8, 'observables': 1, 'DEM_correlation_separators': 6}
Reference files available: {'actual': True, 'PM': True, 'CM': True}


## 5. Construct decoders and check predictions

Construction is excluded from per-shot timings. CM enables correlations both when constructing the matching graph and when decoding. A stored-prediction mismatch means this is a controlled comparison of the stated implementations, not an exact reproduction of the archive's decoders.

In [16]:
build_start = time.perf_counter_ns()
pm = pymatching.Matching.from_detector_error_model(dem, enable_correlations=False)
pm_build_s = (time.perf_counter_ns() - build_start) / 1e9
build_start = time.perf_counter_ns()
cm = pymatching.Matching.from_detector_error_model(dem, enable_correlations=True)
cm_build_s = (time.perf_counter_ns() - build_start) / 1e9
print({'PM_build_seconds': pm_build_s, 'CM_build_seconds': cm_build_s})
assert pm.num_detectors == cm.num_detectors == dem.num_detectors

pm_pred = np.asarray(pm.decode_batch(shots, enable_correlations=False)).reshape(N_SHOTS, -1)
cm_pred = np.asarray(cm.decode_batch(shots, enable_correlations=True)).reshape(N_SHOTS, -1)
assert pm_pred.shape == cm_pred.shape == (N_SHOTS, dem.num_observables)
for name, predicted, reference in [('PM', pm_pred, stored_pm), ('CM', cm_pred, stored_cm)]:
    if reference is not None:
        mismatches = int(np.any(predicted != reference, axis=1).sum())
        print(f'{name} versus stored prediction: {mismatches}/{N_SHOTS} differing shots')
if actual is not None:
    for name, predicted in [('PM', pm_pred), ('CM', cm_pred)]:
        errors = int(np.any(predicted != actual, axis=1).sum())
        print(f'{name} logical error rate: {errors}/{N_SHOTS} = {errors/N_SHOTS:.4%}')

{'PM_build_seconds': 0.00068338, 'CM_build_seconds': 0.000278482}
PM versus stored prediction: 0/10000 differing shots
CM versus stored prediction: 0/10000 differing shots
PM logical error rate: 166/10000 = 1.6600%
CM logical error rate: 166/10000 = 1.6600%


## 6. Time batch throughput

Both calls receive the same `shots` array. Warm-up and prediction checks above are outside the timer. Alternating call order reduces systematic ordering effects. Each timed run returns predictions; do not include DEM parsing, file reading, or decoder construction in these figures.

In [17]:
REPEATS = 5
timing_rows = []
for repetition in range(REPEATS):
    order = [('PM', pm, False), ('CM', cm, True)]
    if repetition % 2:
        order.reverse()
    for name, decoder, correlated in order:
        start = time.perf_counter_ns()
        result = decoder.decode_batch(shots, enable_correlations=correlated)
        elapsed_ns = time.perf_counter_ns() - start
        assert len(result) == N_SHOTS
        timing_rows.append({'configuration': chosen_dir, 'mode': 'batch', 'decoder': name,
                            'repetition': repetition + 1, 'shots': N_SHOTS,
                            'elapsed_s': elapsed_ns / 1e9,
                            'us_per_shot': elapsed_ns / N_SHOTS / 1e3})
batch_runs = pd.DataFrame(timing_rows)
batch_summary = batch_runs.groupby('decoder')['us_per_shot'].agg(['median', 'min', 'max'])
display(batch_summary)
pm_batch_us = batch_summary.loc['PM', 'median']
cm_batch_us = batch_summary.loc['CM', 'median']
print(f'Batch CM/PM cost ratio: {cm_batch_us / pm_batch_us:.3f}x')
print(f'Batch CM minus PM: {cm_batch_us - pm_batch_us:.3f} µs/shot')
print(f'Median throughput: PM {1e6/pm_batch_us:,.0f}, CM {1e6/cm_batch_us:,.0f} shots/s')

,median,min,max
decoder,,,
CM,0.879538,0.852979,0.952822
PM,0.335597,0.332888,0.473588


Batch CM/PM cost ratio: 2.621x
Batch CM minus PM: 0.544 µs/shot
Median throughput: PM 2,979,764, CM 1,136,961 shots/s


## 7. Time sequential per-shot latency

This separately measures Python calls to `decode` one shot at a time on the same subset. It includes Python call overhead for **both** decoders. Use this mode when assessing a router that chooses and decodes each shot immediately; do not substitute batch throughput for online latency.

In [18]:
SINGLE_SHOT_N = min(1_000, N_SHOTS)
single_shots = shots[:SINGLE_SHOT_N]
for decoder, correlated in [(pm, False), (cm, True)]:
    for row in single_shots[:min(100, SINGLE_SHOT_N)]:
        decoder.decode(row, enable_correlations=correlated)
single_rows = []
for repetition in range(REPEATS):
    order = [('PM', pm, False), ('CM', cm, True)]
    if repetition % 2:
        order.reverse()
    for name, decoder, correlated in order:
        start = time.perf_counter_ns()
        for row in single_shots:
            decoder.decode(row, enable_correlations=correlated)
        elapsed_ns = time.perf_counter_ns() - start
        single_rows.append({'configuration': chosen_dir, 'mode': 'sequential', 'decoder': name,
                            'repetition': repetition + 1, 'shots': SINGLE_SHOT_N,
                            'elapsed_s': elapsed_ns / 1e9,
                            'us_per_shot': elapsed_ns / SINGLE_SHOT_N / 1e3})
single_runs = pd.DataFrame(single_rows)
single_summary = single_runs.groupby('decoder')['us_per_shot'].agg(['median', 'min', 'max'])
display(single_summary)
pm_single_us = single_summary.loc['PM', 'median']
cm_single_us = single_summary.loc['CM', 'median']
print(f'Sequential CM/PM cost ratio: {cm_single_us / pm_single_us:.3f}x')
print(f'Sequential CM minus PM: {cm_single_us - pm_single_us:.3f} µs/shot')
runs = pd.concat([batch_runs, single_runs], ignore_index=True)
runs['stim_version'] = stim.__version__
runs['pymatching_version'] = pymatching.__version__
runs['cpu_model'] = CPU_MODEL
runs['archive_md5'] = EXPECTED_MD5
runs['dem_sha256'] = hashlib.sha256(paths['circuit_detector_error_model.dem'].read_bytes()).hexdigest()
output_path = WORK / 'decoder_timing_runs.csv'
runs.to_csv(output_path, index=False)
print('Saved repeated timing measurements:', output_path)

,median,min,max
decoder,,,
CM,8.315803,7.710425,9.844991
PM,7.342225,7.222054,7.896249


Sequential CM/PM cost ratio: 1.133x
Sequential CM minus PM: 0.974 µs/shot
Saved repeated timing measurements: /content/decoder_runtime/decoder_timing_runs.csv


## Interpreting these results

The CM/PM ratio and difference above establish decoder cost **for this configuration and Colab CPU**. Repeat with other distances, bases, and round counts, recording each separately. Decoder construction times are reported but excluded from per-shot cost.

To claim router savings later, align the router's feature representation and decisions with these exact shots. Time feature extraction, model inference, and the selected decoder in the **same mode** as the always-CM baseline. Then compute `100 * (T_always_CM - T_routed) / T_always_CM`. The earlier 44.1% CM usage rate is only an illustrative estimate until decisions on these shots are available.

## 8. Main PM-versus-CM timing benchmark

The saved run above is a pilot on distance 3 with one round. The saved main run below measures a small, fixed set of **surface-code** settings: both bases at distance 3 and 5 rounds, distance 5 and 5 rounds, and distance 5 and 25 rounds. For distance 3 it selects the first chip position in the archive and records its full experiment name. Change `REPRESENTATIVE_SETTINGS` if another setting is needed. Results stay separate by experiment.

Every configuration uses a fixed-seed sample of 10,000 archive shots spread across the full acquisition for **both** decoders and **both** execution modes. The selected archive shot indices are saved in a separate CSV. The notebook checks stored predictions and logical error rates as decoder validation; the result of this section is decoding **time**. Decoder construction, ZIP reading, bit unpacking, and validation are outside the timer. Batch mode times `decode_batch`; sequential mode times one `decode` call per shot. Do not compare a batch time with a sequential time.

Each timing block repeats the same shot set enough times to target at least 0.2 seconds for PM, then uses the **same pass count for CM**. Seven blocks alternate decoder order. Report median and interquartile range; keep all raw timing blocks in the CSV. A later router experiment must use the same execution mode and CPU for a net-savings claim.

In [19]:
import re
REPRESENTATIVE_SETTINGS = [('X', 3, 5), ('Z', 3, 5),
                           ('X', 5, 5), ('Z', 5, 5),
                           ('X', 5, 25), ('Z', 5, 25)]
MAIN_SHOTS = 10_000
MAIN_SAMPLE_SEED = 42
MAIN_REPEATS = 7
MIN_BLOCK_SECONDS = 0.20
MAX_PASSES = 200
name_pattern = re.compile(r'^surface_code_b([XZ])_d(\d+)_r(\d+)_center_\d+_\d+$')
available = {}
for directory in candidates:
    match = name_pattern.fullmatch(directory)
    if match:
        key = (match.group(1), int(match.group(2)), int(match.group(3)))
        available.setdefault(key, []).append(directory)
selected_configurations = []
for setting in REPRESENTATIVE_SETTINGS:
    choices = sorted(available.get(setting, []))
    if not choices:
        raise ValueError(f'Archive has no surface-code experiment for {setting}')
    selected_configurations.append((setting, choices[0]))
print('Selected experiments:')
for setting, directory in selected_configurations:
    print(setting, directory)

Selected experiments:
('X', 3, 5) surface_code_bX_d3_r05_center_3_5
('Z', 3, 5) surface_code_bZ_d3_r05_center_3_5
('X', 5, 5) surface_code_bX_d5_r05_center_5_5
('Z', 5, 5) surface_code_bZ_d5_r05_center_5_5
('X', 5, 25) surface_code_bX_d5_r25_center_5_5
('Z', 5, 25) surface_code_bZ_d5_r25_center_5_5


In [20]:
def read_reference_from_zip(archive, member, indices, observable_count, total_shots):
    if member is None:
        return None
    with archive.open(member) as source:
        all_lines = [line.decode('ascii').strip() for line in source]
    if len(all_lines) != total_shots:
        raise ValueError(f'Reference shot count differs from detector data: {member}')
    lines = [all_lines[index] for index in indices]
    if any(len(line) != observable_count or set(line) - {'0', '1'} for line in lines):
        raise ValueError(f'Invalid or short 01 reference file: {member}')
    return np.asarray([[int(bit) for bit in line] for line in lines], dtype=np.uint8)

def run_decoder(decoder, correlated, mode, data):
    if mode == 'batch':
        return decoder.decode_batch(data, enable_correlations=correlated)
    for row in data:
        decoder.decode(row, enable_correlations=correlated)

def benchmark_mode(data, pm_decoder, cm_decoder, mode):
    if mode not in ('batch', 'sequential'):
        raise ValueError('mode must be batch or sequential')
    decoders = [('PM', pm_decoder, False), ('CM', cm_decoder, True)]
    for _, decoder, correlated in decoders:
        run_decoder(decoder, correlated, mode, data[:min(1_000, len(data))])
    start = time.perf_counter_ns()
    run_decoder(pm_decoder, False, mode, data)
    pilot_seconds = max((time.perf_counter_ns() - start) / 1e9, 1e-9)
    passes = min(MAX_PASSES, max(1, int(np.ceil(MIN_BLOCK_SECONDS / pilot_seconds))))
    records = []
    for repetition in range(MAIN_REPEATS):
        order = decoders if repetition % 2 == 0 else list(reversed(decoders))
        for name, decoder, correlated in order:
            start = time.perf_counter_ns()
            for _ in range(passes):
                run_decoder(decoder, correlated, mode, data)
            elapsed_ns = time.perf_counter_ns() - start
            records.append({'mode': mode, 'decoder': name, 'repetition': repetition + 1,
                            'shots_per_pass': len(data), 'passes': passes,
                            'timed_shots': len(data) * passes,
                            'elapsed_s': elapsed_ns / 1e9,
                            'us_per_shot': elapsed_ns / (len(data) * passes) / 1e3})
    return records

main_timing_rows = []
main_runs_path = WORK / 'decoder_representative_timing_runs.csv'
sample_manifest_rows = []
sample_manifest_path = WORK / 'decoder_sample_indices.csv'
with ZipFile(ARCHIVE) as archive:
    for (basis, distance, rounds), directory in selected_configurations:
        files = by_dir[directory]
        destination = WORK / 'representative' / hashlib.sha256(directory.encode()).hexdigest()[:12]
        destination.mkdir(parents=True, exist_ok=True)
        local_paths = {}
        for basename in ('circuit_detector_error_model.dem', 'detection_events.b8'):
            path = destination / basename
            with archive.open(files[basename]) as source, path.open('wb') as output:
                shutil.copyfileobj(source, output)
            local_paths[basename] = path
        model = stim.DetectorErrorModel.from_file(local_paths['circuit_detector_error_model.dem'])
        packed_case = stim.read_shot_data_file(path=local_paths['detection_events.b8'],
                                               format='b8', num_detectors=model.num_detectors,
                                               bit_packed=True)
        if len(packed_case) < MAIN_SHOTS:
            raise ValueError(f'{directory}: only {len(packed_case)} shots')
        sample_indices = np.sort(np.random.default_rng(MAIN_SAMPLE_SEED).choice(
            len(packed_case), size=MAIN_SHOTS, replace=False))
        sample_manifest_rows.extend({'configuration': directory, 'archive_shot_index': int(index)}
                                    for index in sample_indices)
        pd.DataFrame(sample_manifest_rows).to_csv(sample_manifest_path, index=False)
        data = np.ascontiguousarray(
            np.unpackbits(packed_case[sample_indices], axis=1, bitorder='little')[:, :model.num_detectors],
            dtype=np.uint8)
        if data.shape != (MAIN_SHOTS, model.num_detectors):
            raise ValueError(f'{directory}: detector shape mismatch')
        start = time.perf_counter_ns()
        pm_case = pymatching.Matching.from_detector_error_model(model, enable_correlations=False)
        pm_build_s = (time.perf_counter_ns() - start) / 1e9
        start = time.perf_counter_ns()
        cm_case = pymatching.Matching.from_detector_error_model(model, enable_correlations=True)
        cm_build_s = (time.perf_counter_ns() - start) / 1e9
        pm_case_pred = np.asarray(pm_case.decode_batch(data, enable_correlations=False)).reshape(MAIN_SHOTS, -1)
        cm_case_pred = np.asarray(cm_case.decode_batch(data, enable_correlations=True)).reshape(MAIN_SHOTS, -1)
        if pm_case_pred.shape != cm_case_pred.shape or pm_case_pred.shape[1] != model.num_observables:
            raise ValueError(f'{directory}: observable shape mismatch')
        reference = {}
        for label, basename in [('actual', 'obs_flips_actual.01'),
                                ('PM', 'obs_flips_predicted_by_pymatching.01'),
                                ('CM', 'obs_flips_predicted_by_correlated_matching.01')]:
            reference[label] = read_reference_from_zip(
                archive, files.get(basename), sample_indices, model.num_observables, len(packed_case))
        pm_mismatch = (int(np.any(pm_case_pred != reference['PM'], axis=1).sum())
                       if reference['PM'] is not None else None)
        cm_mismatch = (int(np.any(cm_case_pred != reference['CM'], axis=1).sum())
                       if reference['CM'] is not None else None)
        pm_ler = (float(np.any(pm_case_pred != reference['actual'], axis=1).mean())
                  if reference['actual'] is not None else None)
        cm_ler = (float(np.any(cm_case_pred != reference['actual'], axis=1).mean())
                  if reference['actual'] is not None else None)
        dem_hash = hashlib.sha256(local_paths['circuit_detector_error_model.dem'].read_bytes()).hexdigest()
        details = {'configuration': directory, 'basis': basis, 'distance': distance, 'rounds': rounds,
                   'source_shots': len(packed_case), 'sample_seed': MAIN_SAMPLE_SEED,
                   'sample_indices_sha256': hashlib.sha256(sample_indices.astype('<i8').tobytes()).hexdigest(),
                   'detectors': model.num_detectors, 'observables': model.num_observables,
                   'pm_reference_mismatches': pm_mismatch, 'cm_reference_mismatches': cm_mismatch,
                   'pm_ler_diagnostic': pm_ler, 'cm_ler_diagnostic': cm_ler,
                   'pm_build_s': pm_build_s, 'cm_build_s': cm_build_s,
                   'dem_sha256': dem_hash, 'archive_md5': EXPECTED_MD5,
                   'stim_version': stim.__version__, 'pymatching_version': pymatching.__version__,
                   'cpu_model': CPU_MODEL}
        print('Timing:', directory, 'detectors:', model.num_detectors,
              'reference mismatches PM/CM:', pm_mismatch, cm_mismatch)
        if (pm_mismatch or 0) > 0 or (cm_mismatch or 0) > 0:
            print(f'Reference predictions differ: label this a controlled PyMatching {pymatching.__version__} benchmark.')
        for mode in ('batch', 'sequential'):
            for row in benchmark_mode(data, pm_case, cm_case, mode):
                main_timing_rows.append({**details, **row})
        pd.DataFrame(main_timing_rows).to_csv(main_runs_path, index=False)
main_runs = pd.DataFrame(main_timing_rows)
main_runs.to_csv(main_runs_path, index=False)
print('Saved all repeated timing blocks:', main_runs_path)
print('Saved selected archive shot indices:', sample_manifest_path)

Timing: surface_code_bX_d3_r05_center_3_5 detectors: 40 reference mismatches PM/CM: 0 446
Reference predictions differ: label this a controlled PyMatching 2.4.0 benchmark.
Timing: surface_code_bZ_d3_r05_center_3_5 detectors: 40 reference mismatches PM/CM: 0 381
Reference predictions differ: label this a controlled PyMatching 2.4.0 benchmark.
Timing: surface_code_bX_d5_r05_center_5_5 detectors: 120 reference mismatches PM/CM: 0 488
Reference predictions differ: label this a controlled PyMatching 2.4.0 benchmark.
Timing: surface_code_bZ_d5_r05_center_5_5 detectors: 120 reference mismatches PM/CM: 0 497
Reference predictions differ: label this a controlled PyMatching 2.4.0 benchmark.
Timing: surface_code_bX_d5_r25_center_5_5 detectors: 600 reference mismatches PM/CM: 338 2579
Reference predictions differ: label this a controlled PyMatching 2.4.0 benchmark.
Timing: surface_code_bZ_d5_r25_center_5_5 detectors: 600 reference mismatches PM/CM: 304 2334
Reference predictions differ: label this

In [21]:
summary_rows = []
for (configuration, mode), group in main_runs.groupby(['configuration', 'mode'], sort=False):
    timings = {name: group.loc[group['decoder'] == name, 'us_per_shot'].to_numpy()
               for name in ('PM', 'CM')}
    pm_median, cm_median = (float(np.median(timings[name])) for name in ('PM', 'CM'))
    summary_rows.append({'configuration': configuration, 'mode': mode,
                         'distance': int(group['distance'].iloc[0]),
                         'rounds': int(group['rounds'].iloc[0]),
                         'basis': group['basis'].iloc[0],
                         'detectors': int(group['detectors'].iloc[0]),
                         'shots_per_pass': int(group['shots_per_pass'].iloc[0]),
                         'passes': int(group['passes'].iloc[0]),
                         'pm_median_us': pm_median, 'cm_median_us': cm_median,
                         'pm_iqr_us': float(np.percentile(timings['PM'], 75) - np.percentile(timings['PM'], 25)),
                         'cm_iqr_us': float(np.percentile(timings['CM'], 75) - np.percentile(timings['CM'], 25)),
                         'cm_over_pm': cm_median / pm_median,
                         'cm_minus_pm_us': cm_median - pm_median,
                         'pm_reference_mismatches': group['pm_reference_mismatches'].iloc[0],
                         'cm_reference_mismatches': group['cm_reference_mismatches'].iloc[0],
                         'cpu_model': group['cpu_model'].iloc[0],
                         'pymatching_version': group['pymatching_version'].iloc[0],
                         'stim_version': group['stim_version'].iloc[0]})
main_summary = pd.DataFrame(summary_rows)
display(main_summary[['configuration', 'mode', 'detectors', 'shots_per_pass', 'passes',
                      'pm_median_us', 'cm_median_us', 'pm_iqr_us', 'cm_iqr_us',
                      'cm_over_pm', 'cm_minus_pm_us',
                      'pm_reference_mismatches', 'cm_reference_mismatches']])
main_summary_path = WORK / 'decoder_representative_timing_summary.csv'
main_summary.to_csv(main_summary_path, index=False)
print('Saved summary:', main_summary_path)
print('These results measure decoder time only. Router overhead and savings are not measured here.')

,configuration,mode,detectors,shots_per_pass,passes,pm_median_us,cm_median_us,pm_iqr_us,cm_iqr_us,cm_over_pm,cm_minus_pm_us,pm_reference_mismatches,cm_reference_mismatches
0,surface_code_bX_d3_r05_center_3_5,batch,40,10000,8,2.732800,9.408360,0.866698,3.033645,3.442754,6.675559,0,446
1,surface_code_bX_d3_r05_center_3_5,sequential,40,10000,2,10.293926,14.716867,0.860700,0.498524,1.429665,4.422941,0,446
2,surface_code_bZ_d3_r05_center_3_5,batch,40,10000,7,2.596567,7.089934,0.058220,0.257566,2.730503,4.493368,0,381
3,surface_code_bZ_d3_r05_center_3_5,sequential,40,10000,2,12.749722,23.402565,5.833012,8.066544,1.835535,10.652843,0,381
4,surface_code_bX_d5_r05_center_5_5,batch,120,10000,2,13.809845,35.977909,0.513437,0.396355,2.605236,22.168064,0,488
5,surface_code_bX_d5_r05_center_5_5,sequential,120,10000,1,30.040855,63.046740,9.232509,20.035935,2.098700,33.005884,0,488
6,surface_code_bZ_d5_r05_center_5_5,batch,120,10000,2,14.303856,36.977273,0.160850,0.209458,2.585126,22.673418,0,497
7,surface_code_bZ_d5_r05_center_5_5,sequential,120,10000,1,23.641640,47.055813,11.137895,16.928947,1.990379,23.414173,0,497
8,surface_code_bX_d5_r25_center_5_5,batch,600,10000,1,100.068632,255.494070,0.561566,49.718545,2.553188,155.425438,338,2579
9,surface_code_bX_d5_r25_center_5_5,sequential,600,10000,1,111.631410,271.137563,24.665336,31.932518,2.428864,159.506153,338,2579


Saved summary: /content/decoder_runtime/decoder_representative_timing_summary.csv
These results measure decoder time only. Router overhead and savings are not measured here.


## Interpretation of the saved main run

**Decoder timing result.** On this Colab CPU (Intel Xeon 2.20 GHz, two logical CPUs; Stim 1.16.0 and PyMatching 2.4.0), correlation-enabled PyMatching took longer than standard PyMatching in all six selected configurations. The table gives the range across X and Z bases of the median **CM minus PM** time on the same 10,000 sampled shots. Batch and sequential modes are separate workloads.

| Code setting | Batch extra CM time (µs/shot) | Sequential extra CM time (µs/shot) |
| --- | ---: | ---: |
| Distance 3, 5 rounds | 4.49–6.68 | 4.42–10.65 |
| Distance 5, 5 rounds | 22.17–22.67 | 23.41–33.01 |
| Distance 5, 25 rounds | 155.43–155.70 | 159.51–202.36 |

The larger distance-5, 25-round gap gives **scope for routing to save decoder time** by sending some shots to PM. This is an opportunity, not a measured net saving. For a rough estimate, if a fraction `p_PM` uses PM and average decoder costs are similar on the routed subsets, gross avoided time is `p_PM × (T_CM − T_PM)` per shot. Feature extraction, model inference, and routing overhead must be smaller than that avoided time. The final calculation must use the router's **actual choices on the same shots and CPU**, in the same batch or sequential mode. The 44.1% CM-use rate from a different validation sample is not a measured rate for these configurations.

**Reproduction limit.** The reconstructed CM predictions differ from the archive's stored CM predictions on 381–497 of 10,000 shots at 5 rounds and 2,334–2,579 at distance 5 with 25 rounds. Reconstructed PM matches the archive at 5 rounds but differs on 304–338 shots at distance 5 with 25 rounds. These timings therefore describe the **specified PyMatching 2.4.0 implementations and DEM**, not the exact runtime of the archived Google decoders. A router trained or evaluated using the stored decoder predictions cannot receive a defensible net-savings claim from these timings alone; first align the decoder implementations or evaluate the router against their reconstructed predictions. Some timing interquartile ranges are also wide, so repeat the benchmark on a stable CPU before a final performance claim.